# Запросы на презентации из WildChat

Отбираем из WildChat русские реплики пользователей с просьбой сделать презентацию.
Это сырьё для внеучебных и прикладных заданий: тему и аудиторию из запросов выделяем потом.

- Набор: `allenai/WildChat-1M`, лицензия ODC-BY; читаем потоково, целиком не скачиваем.
- Берём реплики пользователя на русском, где есть просьба о презентации, в том числе с опечатками (см. `PATTERN`).
- Токсичные диалоги и диалоги с удалёнными персональными данными пропускаем.
- Сохраняем только текст запроса, id диалога и номер реплики; ответы модели и сведения о пользователе не храним.

GPU не нужен: запускается и на своём компьютере (нужен интернет и пакет `datasets`), и на Kaggle.
Если набор требует принять условия на Hugging Face, токен берётся из переменной окружения `HF_TOKEN`
или из секретов Kaggle под тем же именем (либо после `huggingface-cli login`).

Результат: `presentation_requests.jsonl` (одна строка — один запрос) и `presentation_requests_stats.json`.
Строки дописываются сразу, поэтому при обрыве найденное сохраняется; повторный запуск начинает файл заново.

In [1]:
import json, os, re, time
from pathlib import Path

from datasets import load_dataset

DATASET = "allenai/WildChat-1M"
# На Kaggle пишем в /kaggle/working, на своём компьютере — рядом с ноутбуком, в ../data/tasks.
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../data/tasks")
OUT = OUT_DIR / "presentation_requests.jsonl"
STATS = OUT_DIR / "presentation_requests_stats.json"
# Слова, по которым узнаём просьбу о презентации, с частыми опечатками и разговорными формами:
# «призентация», «презинтация», «презетация», «прозентация», «преза», «презу», «презка»,
# «слайды» (но не «слайдер»), PowerPoint и его русские написания, pptx, английские presentation/slides.
# Не ловим «процент», «проза», «президент», «презент», «презумпция», «претензия».
PATTERN = re.compile(
    r"пр[еиаоэ]з[еиаоэ]?н?т[аоеи]?ц"
    r"|\bпрез(?:а|у|ы|е|ой|к[аиуе]|ку)\b"
    r"|слайд(?!ер)"
    r"|power\s?point|п[ао][ув]?[эе]р\s?п[оа]инт|\bpptx?\b"
    r"|presentation|\bslides?\b",
    re.IGNORECASE,
)

token = os.environ.get("HF_TOKEN")
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # набор открытый, или токен сохранён через huggingface-cli login

c:\Users\Дарья\Desktop\D_agents\Diplom\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset(DATASET, split="train", streaming=True, token=token)

found = 0
stats = {"source": DATASET, "conversations": 0, "russian": 0, "skipped_toxic": 0, "skipped_redacted": 0}
start = time.time()
with OUT.open("w", encoding="utf-8") as f:
    for conv in ds:
        stats["conversations"] += 1
        if stats["conversations"] % 50000 == 0:
            print(f"{stats['conversations']} диалогов, найдено {found}, {time.time() - start:.0f} с")
        if conv.get("language") != "Russian":
            continue
        stats["russian"] += 1
        if conv.get("toxic"):
            stats["skipped_toxic"] += 1
            continue
        if conv.get("redacted"):
            stats["skipped_redacted"] += 1
            continue
        user_turn = 0
        for msg in conv["conversation"]:
            if msg["role"] != "user":
                continue
            text = msg["content"] or ""
            m = PATTERN.search(text)
            if m:
                row = {"id": conv["conversation_hash"], "turn": user_turn, "match": m.group(0), "text": text}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                found += 1
            user_turn += 1

stats["found"] = found
STATS.write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding="utf-8")
print(stats)

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/allenai/WildChat-1M/resolve/7d6490e462285cf85d91eabea0f9a954fbddcd1f/data/train-00000-of-00014.parquet
Retrying in 1s [Retry 1/5].


50000 диалогов, найдено 93, 112 с


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/allenai/WildChat-1M/resolve/7d6490e462285cf85d91eabea0f9a954fbddcd1f/data/train-00001-of-00014.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/allenai/WildChat-1M/resolve/7d6490e462285cf85d91eabea0f9a954fbddcd1f/data/train-00001-of-00014.parquet
Retrying in 2s [Retry 2/5].


100000 диалогов, найдено 210, 420 с
150000 диалогов, найдено 381, 499 с
200000 диалогов, найдено 432, 589 с
250000 диалогов, найдено 507, 685 с
300000 диалогов, найдено 530, 758 с
350000 диалогов, найдено 556, 826 с
400000 диалогов, найдено 590, 899 с
450000 диалогов, найдено 652, 967 с
500000 диалогов, найдено 726, 1024 с
550000 диалогов, найдено 821, 1105 с
600000 диалогов, найдено 915, 1237 с
650000 диалогов, найдено 1083, 1368 с
700000 диалогов, найдено 1281, 1516 с
750000 диалогов, найдено 1350, 1631 с
800000 диалогов, найдено 1505, 1757 с
{'source': 'allenai/WildChat-1M', 'conversations': 837989, 'russian': 87127, 'skipped_toxic': 0, 'skipped_redacted': 196, 'found': 1691}


In [3]:
# Несколько случайных запросов, чтобы оценить, что нашлось.
import random

rows = [json.loads(line) for line in OUT.open(encoding="utf-8")]
random.seed(0)
for r in random.sample(rows, min(15, len(rows))):
    print(r["id"], r["turn"], "|", r["text"][:300].replace("\n", " "))
    print("---")

a9d9de3fb7dd4d83a043724528be2f2f 1 | опиши подробно Slide 7: Заключение - Повторение основной информации о каждом типе аквальных четвертичных отложений - Подчеркивание их важности для наук о Земле и экоигологии 
---
c8cab603309fd7347b8e558ae1986b93 12 | Напиши профессионально краткое описание для создания презентации под существующий культурный объект Louis Vuitton Foundation, отразив в дизайне его идейную концепцию, эскизность и воздушные формы через журнальный стиль
---
84ef559c3478772aeaa445a2c1962c97 3 | Я сделал презентацию на тему: “Протоколы и особенности функционирования VPN”. Напиши защитное слово для слайда "Особенности функционирования VPN" 
---
e936a1e3c8fc6a5b1a16dae1699f92b8 3 | Составь текст для слайда 13. Основные принципы поиска
---
6347abab40a1302871dbde8749fbc459 0 | Ты методист университета нутрициологии. Подготовь презентацию по теме "Лабораторная и инструментальная диагностика гиперхолестеринемии"
---
fa945b6346b6ecdbdb24627a665044a2 6 | написать рецензию отзыв по